## Agentic AI - Complete Practice Notebook (VS Code)
Welcome to the consolidated Agentic AI practice notebook. This file combines all function calling, ReAct loops, stateful graphs, and memory checkpointer exercises.

### <a id="table-of-contents"></a>📋 Table of Contents
* [Chapter 1: Introduction to Agents](#chapter-1-introduction-to-agents)
* [Chapter 2: Tool Use & Function Calling](#chapter-2-tool-use-and-function-calling)
* [Chapter 3: Building a ReAct Agent](#chapter-3-building-a-react-agent)
* [Chapter 4: Stateful Travel Assistant Project](#chapter-4-stateful-travel-assistant-project)
* [Chapter 5: Resume Analyzer & Interview Planner capstone workflow](#chapter-5-resume-analyzer-and-interview-planner-capstone-workflow)

### 🛠️ Setup Environment & Load API Keys

In [ ]:
# Load local environment variables from .env
import os
from dotenv import load_dotenv
load_dotenv()

### <a id="chapter-1-introduction-to-agents"></a>📖 Chapter 1: Introduction to Agents
Agents use LLMs as reasoning engines to choose actions, evaluate observations, and loop dynamically until a target goal is achieved.

[Back to Top](#table-of-contents)

### <a id="chapter-2-tool-use-and-function-calling"></a>📖 Chapter 2: Tool Use & Function Calling

#### 2.1 Declaring Tools with @tool Decorator

In [ ]:
from langchain_core.tools import tool

@tool
def calculate_perimeter(width: float, height: float) -> float:
    """Calculates the perimeter of a rectangle given width and height."""
    return 2 * (width + height)

print("Tool Name:", calculate_perimeter.name)
print("Tool Description:", calculate_perimeter.description)

#### 2.2 Binding Tools to Chat Models

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")
llm_with_tools = llm.bind_tools([calculate_perimeter])

response = llm_with_tools.invoke("What is the perimeter of a box with width 5 and height 10?")
print("Tool Calls Requested:", response.tool_calls)

[Back to Top](#table-of-contents)

### <a id="chapter-3-building-a-react-agent"></a>📖 Chapter 3: Building a ReAct Agent

#### 3.1 Initializing Prebuilt ReAct Agent in LangGraph

In [ ]:
from langgraph.prebuilt import create_react_agent

# Compose ReAct Loop Agent
agent_executor = create_react_agent(llm, tools=[calculate_perimeter])

state = agent_executor.invoke({"messages": [("user", "Calculate the perimeter of a yard that is 12m wide and 15m long.")]})
print("Final Response:", state["messages"][-1].content)

[Back to Top](#table-of-contents)

### <a id="chapter-4-stateful-travel-assistant-project"></a>📖 Chapter 4: Stateful Travel Assistant Project

#### 4.1 Define Custom State, Nodes, and Graph Construction

In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

class TravelState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    destination: str
    weather_info: str
    budget_estimate: str

def weather_agent(state: TravelState):
    # Mock node updating destination & weather
    return {
        "messages": [AIMessage(content="Checked weather forecast.")],
        "destination": "Tokyo",
        "weather_info": "18°C, Cloudy with light rain."
    }

def budget_agent(state: TravelState):
    # Mock node calculating budget
    return {
        "messages": [AIMessage(content="Calculated travel estimates.")],
        "budget_estimate": "3-day trip budget for Tokyo: $600."
    }

# Assemble custom stateful graph
builder = StateGraph(TravelState)
builder.add_node("weather_agent", weather_agent)
builder.add_node("budget_agent", budget_agent)
builder.add_edge(START, "weather_agent")
builder.add_edge("weather_agent", "budget_agent")
builder.add_edge("budget_agent", END)

memory = MemorySaver()
travel_app = builder.compile(checkpointer=memory)

# Run custom session
config = {"configurable": {"thread_id": "test-session"}}
final_state = travel_app.invoke({"messages": [HumanMessage(content="Plan Tokyo.")]}, config)
print("Final Budget:", final_state["budget_estimate"])

[Back to Top](#table-of-contents)

### <a id="chapter-5-resume-analyzer-and-interview-planner-capstone-workflow"></a>📖 Chapter 5: Resume Analyzer & Interview Planner Capstone Workflow

#### 5.1 Define Stateful Resume Analysis Schema

In [ ]:
class AnalysisState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    resume_text: str
    skills_match: str
    interview_schedule: str

def analyzer_node(state: AnalysisState):
    # Mock resume parsing
    return {
        "messages": [AIMessage(content="Analyzed resume skills.")],
        "skills_match": "Python: Strong Match, Docker: Missing."
    }

def planner_node(state: AnalysisState):
    # Mock interview planner
    return {
        "messages": [AIMessage(content="Scheduled interview.")],
        "interview_schedule": "Monday at 10 AM, covering Python fundamentals."
    }

# Build Resume Analyzer graph
resume_builder = StateGraph(AnalysisState)
resume_builder.add_node("analyzer", analyzer_node)
resume_builder.add_node("planner", planner_node)
resume_builder.add_edge(START, "analyzer")
resume_builder.add_edge("analyzer", "planner")
resume_builder.add_edge("planner", END)

resume_app = resume_builder.compile(checkpointer=MemorySaver())

result_state = resume_app.invoke({"resume_text": "Python coder profile."}, {"configurable": {"thread_id": "session-123"}})
print("Interview Details:", result_state["interview_schedule"])